In [1]:
# Cell 1/7 — Imports + file paths (edit paths, then run)

import os                                  # file paths
import numpy as np                         # arrays
import pandas as pd                        # metadata table
import zarr                                # read Zarr store
from tqdm.auto import tqdm                 # progress bar

from sklearn.ensemble import RandomForestClassifier  # random forest (classification)
from sklearn.preprocessing import LabelEncoder       # convert hue strings -> ints
from sklearn.metrics import confusion_matrix, classification_report  # evaluation

# ---- EDIT THESE TWO PATHS ----
ZARR_PATH = "/Users/kate/Documents/retina-model/notebooks/export_retina_20260119_105329/dataset.zarr"   
META_CSV  = "/Users/kate/Documents/retina-model/notebooks/export_retina_20260119_105329/metadata.csv"   


In [2]:
# Cell 2/7 — Open the dataset + load metadata

root = zarr.open(ZARR_PATH, mode="r")      # open Zarr store (read-only)

imgs = root["imgs"]                        # (N,H,W,3) stimulus images
outs = root["outs"]                        # (N,K,H,W) raw responses (saved for completeness)
outs_fill = root["outs_fill"]              # (N,K,H,W) filled responses (main signal)

keys_order = list(root.attrs["keys_order"])# list of K response-map names (channel order)
meta = pd.read_csv(META_CSV)               # metadata table (one row per stimulus)

# Quick sanity checks
N = outs_fill.shape[0]                     # number of stimuli
K = outs_fill.shape[1]                     # number of response channels
H, W = outs_fill.shape[2], outs_fill.shape[3]
print("H,W:", H, W)
print("N stimuli:", N)
print("K channels:", K)
print("keys_order:", keys_order)
print(meta.head())


H,W: 1000 1000
N stimuli: 250
K channels: 8
keys_order: ['l_on', 'l_off', 'm_on', 'm_off', 'l_h2_on', 'm_h2_on', 'l_h2_off', 'm_h2_off']
   stim_id  hue  intensity  kind  bg_r  bg_g  bg_b  center_r  center_g  \
0        0  red       0.02  flat   0.5   0.5   0.5      0.02       0.0   
1        1  red       0.04  flat   0.5   0.5   0.5      0.04       0.0   
2        2  red       0.06  flat   0.5   0.5   0.5      0.06       0.0   
3        3  red       0.08  flat   0.5   0.5   0.5      0.08       0.0   
4        4  red       0.10  flat   0.5   0.5   0.5      0.10       0.0   

   center_b  
0       0.0  
1       0.0  
2       0.0  
3       0.0  
4       0.0  


In [3]:
# Cell 3/7 — Define the 3 channel-sets + helper functions

# Channel subsets you asked for
keys_all8 = ["l_on","l_off","m_on","m_off","l_h2_on","m_h2_on","l_h2_off","m_h2_off"]
keys_classic4 = ["l_on","l_off","m_on","m_off"]
keys_h2_4 = ["l_h2_on","m_h2_on","l_h2_off","m_h2_off"]

# Map key name -> channel index in the saved arrays
key_to_idx = {k: i for i, k in enumerate(keys_order)}

#def normalize_map(A):
#    """Normalize one 2D map so intensity scale is reduced (robust, per-map)."""
#    A = A.astype(np.float32, copy=False)                 # ensure float32
#    A = A - np.median(A)                                 # remove offset (centers distribution)
#    s = np.percentile(np.abs(A), 99)                     # robust scale (ignores extreme outliers)
#    if s > 0:                                            # avoid divide-by-zero
#        A = A / s                                        # scale to comparable magnitude
#    return A                                             # return normalized map

def normalize_map(A, eps=1e-12):
    """Z-score one 2D map: subtract mean, divide by std (per-map)."""
    A = A.astype(np.float32, copy=False)          # ensure float32
    mu = float(A.mean())                           # map mean
    sd = float(A.std())                            # map standard deviation
    if sd < eps:                                   # avoid divide-by-zero / tiny std
        return (A - mu)                             # if nearly constant, just center
    return (A - mu) / sd                            # z-scored map

#def pool_to_64(A, crop=256):
#    """
#    Center-crop to (crop x crop), then average-pool to (64 x 64).
#    With crop=256, pooling is 4x4 blocks -> 64.
#    """
#    H, W = A.shape                                       # current shape (should be 1000x1000)
#    cy, cx = H // 2, W // 2                              # center coordinates
#    half = crop // 2                                     # half crop size
#    A = A[cy-half:cy+half, cx-half:cx+half]              # center crop -> (crop,crop)
#    A = A.reshape(64, crop//64, 64, crop//64).mean(axis=(1, 3))  # avg-pool -> (64,64)
#    return A.astype(np.float32)                          # return float32

def build_X_for_keys(keys_subset, desc="building X"):
    """
    Build feature matrix X for a chosen subset of channels.
    Each stimulus becomes one feature vector by:
      (1) selecting channels
      (2) per-map normalize
      (3) flatten and concatenate across channels
    """
    idxs = [key_to_idx[k] for k in keys_subset]           # channel indices for this subset
    X = np.zeros((N, len(idxs) * H * W), dtype=np.float32)  # preallocate features

    for i in tqdm(range(N), desc=desc):                   # loop over stimuli
        feat_parts = []                                   # collect per-channel features
        for ch in idxs:                                   # loop over selected channels
            A = np.asarray(outs_fill[i, ch, :, :], dtype=np.float32)                    # read one filled map (H,W)
#            A = normalize_map(A)                          # normalize
#            A = pool_to_64(A, crop=256)                   # downsample to (64,64)
            feat_parts.append(A.ravel())                  # flatten and store
        X[i] = np.concatenate(feat_parts, axis=0)         # concatenate all channels
    return X                                              # (N, n_features)


In [4]:
# Cell 4/7 — Build X matrices and y labels (hue classification target)

# Build features for each input option
X_all8 = build_X_for_keys(keys_all8,     desc="X_all8 (8 channels)")
X_cl4  = build_X_for_keys(keys_classic4, desc="X_classic4 (4 channels)")
X_h2   = build_X_for_keys(keys_h2_4,     desc="X_h2 (4 channels)")

# Labels: hue strings -> integer classes
le = LabelEncoder()                                      # encoder for hue labels
y_str = meta["hue"].astype(str).values                   # hue column as strings
y = le.fit_transform(y_str)                              # numeric labels 0..C-1

print("Class labels:", list(le.classes_))                 # show hue class names
print("X_all8 shape:", X_all8.shape)
print("X_cl4  shape:", X_cl4.shape)
print("X_h2   shape:", X_h2.shape)


X_all8 (8 channels):   0%|          | 0/250 [00:00<?, ?it/s]

X_classic4 (4 channels):   0%|          | 0/250 [00:00<?, ?it/s]

X_h2 (4 channels):   0%|          | 0/250 [00:00<?, ?it/s]

Class labels: ['blue', 'green', 'red', 'white', 'yellow']
X_all8 shape: (250, 8000000)
X_cl4  shape: (250, 4000000)
X_h2   shape: (250, 4000000)


In [5]:
# Cell 5/7 — random 75/25 holdout split (manual, seed=0)

np.random.seed(0)                                        # tutorial-style fixed seed

all_idx = np.arange(N)                                   # all stimulus indices
n_train = int(0.75 * N)                                  # 75% train

train_idx = np.random.choice(all_idx, size=n_train, replace=False)  # random train set
test_idx = np.setdiff1d(all_idx, train_idx)              # remaining 25% test set

# (Optional) sort for nicer reproducibility/printing
train_idx = np.sort(train_idx)                           # sort indices
test_idx = np.sort(test_idx)

print("Train N:", len(train_idx), " Test N:", len(test_idx))


Train N: 187  Test N: 63


In [6]:
# Cell 6/7 — Train 3 RandomForestClassifier models 

MAX_DEPTH = 5                                             # fixed (as requested)
N_EST = 200                                               # number of trees (reasonable default)

def fit_rf(X, name):
    """Fit one RF classifier on the tutorial split."""
    rf = RandomForestClassifier(
        n_estimators=N_EST,                               # number of trees
        max_depth=MAX_DEPTH,                              # fixed depth
        random_state=0,                                   # fixed randomness (tutorial style)
        n_jobs=-1                                         # use all CPU cores
    )
    rf.fit(X[train_idx], y[train_idx])                    # train
    return rf                                             # return trained model

rf_all8 = fit_rf(X_all8, "all8")                          # model using all 8 channels
rf_cl4  = fit_rf(X_cl4,  "classic4")                      # model using classical 4
rf_h2   = fit_rf(X_h2,   "h2_4")                          # model using H2+ 4

rf_all8 = fit_rf(X_all8, "all8")
print("ALL8  train:", rf_all8.score(X_all8[train_idx], y[train_idx]))
print("ALL8  test :", rf_all8.score(X_all8[test_idx],  y[test_idx]))

rf_cl4 = fit_rf(X_cl4, "classic4")
print("CL4   train:", rf_cl4.score(X_cl4[train_idx], y[train_idx]))
print("CL4   test :", rf_cl4.score(X_cl4[test_idx],  y[test_idx]))

rf_h2 = fit_rf(X_h2, "h2_4")
print("H2    train:", rf_h2.score(X_h2[train_idx], y[train_idx]))
print("H2    test :", rf_h2.score(X_h2[test_idx],  y[test_idx]))


ALL8  train: 1.0
ALL8  test : 1.0
CL4   train: 0.9090909090909091
CL4   test : 0.8412698412698413
H2    train: 1.0
H2    test : 1.0


In [7]:
# Cell 7/7 — Evaluate (+ confusion matrices)

def evaluate(rf, X, title):
    """Print tutorial-style .score plus useful classification diagnostics."""
    acc = rf.score(X[test_idx], y[test_idx])              # .score = accuracy for classifier
    print("\n" + "="*70)
    print(title)
    print(f"Test accuracy (.score): {acc:.4f}")

    y_pred = rf.predict(X[test_idx])                      # predicted classes
    cm = confusion_matrix(y[test_idx], y_pred)            # confusion matrix
    print("Confusion matrix (rows=true, cols=pred):")
    print(cm)

    # Optional: more detailed report (precision/recall/F1 per hue)
    print("\nClassification report:")
    print(classification_report(y[test_idx], y_pred, target_names=le.classes_))

evaluate(rf_all8, X_all8, "RF hue prediction — ALL 8 channels")
evaluate(rf_cl4,  X_cl4,  "RF hue prediction — CLASSICAL 4 channels")
evaluate(rf_h2,   X_h2,   "RF hue prediction — H2+ 4 channels")



RF hue prediction — ALL 8 channels
Test accuracy (.score): 1.0000
Confusion matrix (rows=true, cols=pred):
[[11  0  0  0  0]
 [ 0 12  0  0  0]
 [ 0  0 12  0  0]
 [ 0  0  0 13  0]
 [ 0  0  0  0 15]]

Classification report:
              precision    recall  f1-score   support

        blue       1.00      1.00      1.00        11
       green       1.00      1.00      1.00        12
         red       1.00      1.00      1.00        12
       white       1.00      1.00      1.00        13
      yellow       1.00      1.00      1.00        15

    accuracy                           1.00        63
   macro avg       1.00      1.00      1.00        63
weighted avg       1.00      1.00      1.00        63


RF hue prediction — CLASSICAL 4 channels
Test accuracy (.score): 0.8413
Confusion matrix (rows=true, cols=pred):
[[11  0  0  0  0]
 [ 0 11  0  1  0]
 [ 0  0 12  0  0]
 [ 0  3  0  8  2]
 [ 0  1  0  3 11]]

Classification report:
              precision    recall  f1-score   support

    

In [13]:
# --- Build df_wrong for CLASSICAL-4 (run right after you compute y_pred for rf_cl4) ---

y_true = y[test_idx]
y_pred = rf_cl4.predict(X_cl4[test_idx])

wrong = (y_pred != y_true)
wrong_stim_id = test_idx[wrong]

df_wrong = meta.iloc[wrong_stim_id].copy()
df_wrong["stim_id"] = wrong_stim_id
df_wrong["true_hue"] = le.inverse_transform(y_true[wrong])
df_wrong["pred_hue"] = le.inverse_transform(y_pred[wrong])

print("Num wrong (classic4):", wrong.sum(), "out of", len(test_idx))
df_wrong.head()


import numpy as np
import pandas as pd

# ---- 1) infer hue from an RGB triplet (expects your pure-hue construction)
def infer_hue_from_rgb(rgb, tol=1e-6):
    rgb = np.asarray(rgb, float)
    on = tuple((rgb > tol).astype(int))  # e.g. (1,0,0)
    mapping = {(1,0,0):"red",(0,1,0):"green",(0,0,1):"blue",(1,1,0):"yellow",(1,1,1):"white"}
    return mapping.get(on, "unknown")

# ---- 2) pick the disk center pixel from each saved image
Himg, Wimg = imgs.shape[1], imgs.shape[2]          # imgs: (N, H, W, 3)
cy, cx = Himg // 2, Wimg // 2

# center RGB for every stimulus (float)
center_rgb = np.asarray(imgs[:, cy, cx, :], dtype=float)  # (N,3)

# ---- 3) inferred hue + intensity from center pixel
inferred_hue = [infer_hue_from_rgb(rgb) for rgb in center_rgb]
inferred_intensity = center_rgb.max(axis=1)  # for your construction, intensity = max channel at center

# ---- 4) compare to metadata
df_check = meta.copy()
df_check["inferred_hue"] = inferred_hue
df_check["inferred_intensity"] = inferred_intensity

# hue mismatches
bad_hue = df_check[df_check["hue"].astype(str) != df_check["inferred_hue"].astype(str)]
print("Hue mismatches:", len(bad_hue))
display(bad_hue.head(10))

# intensity mismatches (allow tiny float tolerance)
if "intensity" in df_check.columns:
    bad_int = df_check[np.abs(df_check["intensity"].astype(float) - df_check["inferred_intensity"]) > 1e-6]
    print("Intensity mismatches:", len(bad_int))
    display(bad_int.head(10))



# assumes you already built df_wrong for the CLASSIC-4 model (misclassified test items)

# 1) Count mistakes by true hue
print(df_wrong["true_hue"].value_counts())

# 2) Mistakes by (true hue, intensity)
err_by_hi = (df_wrong
             .groupby(["true_hue", "intensity"])
             .size()
             .reset_index(name="n_errors")
             .sort_values(["true_hue","intensity"]))
err_by_hi.head(30)

# 3) Compare error rate per hue (errors / total in test for that hue)
df_test = meta.iloc[test_idx].copy()
df_test["true_hue"] = le.inverse_transform(y[test_idx])
df_test["is_error"] = False
df_test.loc[df_wrong["stim_id"].values, "is_error"] = True

err_rate = df_test.groupby("true_hue")["is_error"].mean().sort_values(ascending=False)
print(err_rate)


Num wrong (classic4): 10 out of 63
Hue mismatches: 0


,stim_id,hue,intensity,kind,bg_r,bg_g,bg_b,center_r,center_g,center_b,inferred_hue,inferred_intensity


Intensity mismatches: 0


,stim_id,hue,intensity,kind,bg_r,bg_g,bg_b,center_r,center_g,center_b,inferred_hue,inferred_intensity


true_hue
white     5
yellow    4
green     1
Name: count, dtype: int64
true_hue
white     0.384615
yellow    0.266667
green     0.083333
blue      0.000000
red       0.000000
Name: is_error, dtype: float64
